# Updated EDA: Cascading Flow Analysis — Bivariate Framework
**Two-Period Comparison (2011 & 2021) with CFI Churn, CFI Rate, and Cascade Direction Ratio**

### Metrics
| Metric | Formula | Captures |
|--------|---------|----------|
| **Net Cascade** | `Inflow_W − Outflow_P` | Directional balance (positive = net gentrification pressure) |
| **CFI Churn** | `Inflow_W + Outflow_P` | Total cascade *intensity* — volume of displacement activity regardless of direction |
| **CFI Rate** | `(Inflow_W × Outflow_P) / Total_Migration` | Normalised interaction (high only when both co-occur relative to turnover) |
| **CDR** *(new)* | `Inflow_W / CFI_Churn` | Cascade *direction* — 1 = pure gentrification pressure, 0 = pure displacement, 0.5 = balanced |
| **% Inflow Wealthier** | `Inflow_W / Total_Inflow × 100` | Share of arrivals from wealthier areas |

### Analytical Framework
CFI Churn measures *intensity* but is directionless. This notebook develops a **bivariate framework** that decomposes
CFI Churn into its two components (Inflow_Wealthier × Outflow_Poorer) and introduces the **Cascade Direction Ratio (CDR)**
to give CFI Churn its directional context. The CDR turns out to be the **strongest single predictor of deprivation change** (r = −0.48).

### Data Source
`msoa_cascade_features_20260515.csv` — 983 London MSOAs.

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)

# Load the pre-computed cascade features
df = pd.read_csv('outputs/msoa_cascade_features_20260515.csv')

# Compute deltas
df['Delta_Net_Cascade'] = df['Net_Cascade_21'] - df['Net_Cascade_11']
df['Delta_CFI_Churn'] = df['CFI_Churn_21'] - df['CFI_Churn_11']
df['Delta_CFI_Rate'] = df['CFI_Rate_21'] - df['CFI_Rate_11']
df['Delta_Pct_Inflow_Wealthier'] = df['Pct_Inflow_Wealthier_21'] - df['Pct_Inflow_Wealthier_11']

# Cascade Direction Ratio: CDR = Inflow_W / CFI_Churn
for suffix in ['_11','_21']:
    df[f'CDR{suffix}'] = np.where(
        df[f'CFI_Churn{suffix}'] > 0,
        df[f'Inflow_Wealthier{suffix}'] / df[f'CFI_Churn{suffix}'],
        np.nan
    )

print(f'MSOAs: {len(df)}')
print(f'Boroughs: {df["ladnm"].nunique()}')
df.head()

---
## 2. Distribution Comparison: 2011 vs 2021

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = [
    ('Net_Cascade', 'Net Cascade (Inflow_W − Outflow_P)'),
    ('CFI_Churn', 'CFI Churn (Inflow_W + Outflow_P)'),
    ('CFI_Rate', 'CFI Rate (Inflow_W × Outflow_P / Total_Mig)'),
    ('Pct_Inflow_Wealthier', '% Inflow from Wealthier Areas'),
]

for ax, (base, title) in zip(axes.flat, metrics):
    c11, c21 = f'{base}_11', f'{base}_21'
    ax.hist(df[c11], bins=40, alpha=0.5, color='#4575b4', label='2011', edgecolor='white')
    ax.hist(df[c21], bins=40, alpha=0.5, color='#d73027', label='2021', edgecolor='white')
    ax.axvline(df[c11].median(), color='#4575b4', ls='--', lw=1.5, label=f'2011 median: {df[c11].median():.0f}')
    ax.axvline(df[c21].median(), color='#d73027', ls='--', lw=1.5, label=f'2021 median: {df[c21].median():.0f}')
    ax.set_xlabel(title); ax.set_ylabel('Number of MSOAs'); ax.legend(fontsize=9)

fig.suptitle('Distribution of Cascade Metrics: 2011 vs 2021 (N=983 London MSOAs)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/fig1_metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Correlation Heatmap: All Metrics

In [ ]:
corr_cols = [
    'CFI_Churn_11','CFI_Rate_11','Net_Cascade_11','Pct_Inflow_Wealthier_11',
    'CFI_Churn_21','CFI_Rate_21','Net_Cascade_21','Pct_Inflow_Wealthier_21',
    'Delta_CFI_Churn','Delta_CFI_Rate','Delta_Net_Cascade','Delta_Pct_Inflow_Wealthier',
    'IMD_Change'
]

fig, ax = plt.subplots(figsize=(14, 11))
cmat = df[corr_cols].corr()
sns.heatmap(cmat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Correlation Matrix: All Cascade Metrics & IMD Change', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/fig2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Initial Validation: All Metrics vs IMD Change

In [ ]:
print('=== Pearson Correlations with IMD Change (2010→2019) ===')
print('  (negative r = metric rises where deprivation fell = expected gentrification signal)\n')

results = []
for col, label in [
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Pct_Inflow_Wealthier_11', '% Inflow Wealthier 2011'),
    ('Pct_Inflow_Wealthier_21', '% Inflow Wealthier 2021'),
]:
    valid = df[[col, 'IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Change'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    results.append({'Metric': label, 'r': r, 'p': p, 'sig': sig})
    print(f'  {label:35s}  r = {r:+.3f}  p = {p:.2e}  {sig}')

##### Initial Findings
- **Net Cascade** is the strongest single predictor (r ≈ −0.44).
- **CFI Churn** is weaker (r ≈ −0.30) because it has **no direction** — it treats gentrification pressure identically to displacement.
- **CFI Rate** does not validate (r ≈ −0.05, ns) — normalisation removes the signal.

**The problem with CFI Churn**: an MSOA with 500 wealthier inflow + 100 poorer outflow has the same churn (600) as one with 100 inflow + 500 outflow. The solution is to **decompose churn into its directional components**.

---
## 5. Cascade Metrics by Wealth Decile: 2011 vs 2021

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
x = np.arange(1, 11)
width = 0.35

for ax, base, title in [
    (axes[0,0], 'CFI_Churn', 'CFI Churn (Additive)'),
    (axes[0,1], 'CFI_Rate', 'CFI Rate (Normalised)'),
    (axes[1,0], 'Net_Cascade', 'Net Cascade (Directional)'),
    (axes[1,1], 'Pct_Inflow_Wealthier', '% Inflow from Wealthier'),
]:
    means = df.groupby('Wealth_Decile').agg(
        y11=(f'{base}_11', 'mean'), y21=(f'{base}_21', 'mean'))
    ax.bar(x - width/2, means['y11'], width, label='2011', color='#4575b4', edgecolor='white')
    ax.bar(x + width/2, means['y21'], width, label='2021', color='#d73027', edgecolor='white')
    ax.set_xlabel('Wealth Decile (1=most deprived, 10=wealthiest)')
    ax.set_ylabel(f'Mean {title}'); ax.set_title(f'{title} by Decile: 2011 vs 2021')
    ax.set_xticks(x); ax.legend()
    if 'Net' in base: ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig('outputs/fig4_metrics_by_decile.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spearman monotonicity tests
print('=== Spearman Rank Correlations (Decile means vs Decile rank) ===\n')
for col, label in [
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
]:
    means = df.groupby('Wealth_Decile')[col].mean()
    rho, p = stats.spearmanr(means.index, means.values)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  Decile vs {label:25s}: ρ = {rho:+.3f}, p = {p:.4f} {sig}')

##### Decile Patterns
- **Net Cascade** shows a near-perfect monotonic decline (ρ = −0.99***): strong positive values in deprived deciles, negative in wealthy deciles.
- **CFI Churn** has an inverted-U shape: peaks at deciles 2–5, zero at extremes. This is by construction — decile 1 has no "wealthier" inflow possible, decile 10 has no "poorer" outflow.
- **CFI Rate** peaks mid-distribution and is not monotonic (ρ = −0.24, ns).
- The inverted-U shape explains why CFI Churn has weaker validation: it rises in both deprived *and* transitional areas.

---
# BIVARIATE FRAMEWORK

The sections below decompose CFI Churn into its directional components and introduce the Cascade Direction Ratio.

---
## 6. Bivariate Quadrant Analysis: Gentrification Pressure × Displacement Yield

By plotting the two components of CFI Churn separately — **Inflow from Wealthier** (x-axis)
vs **Outflow to Poorer** (y-axis) — we can classify MSOAs into four quadrants that distinguish
the *type* of cascade activity happening in each area.

In [ ]:
# Quadrant classification
for suffix, year in [('_11', '2011'), ('_21', '2021')]:
    iw_med = df[f'Inflow_Wealthier{suffix}'].median()
    op_med = df[f'Outflow_Poorer{suffix}'].median()
    
    def quad(row, s=suffix, im=iw_med, om=op_med):
        hi_in = row[f'Inflow_Wealthier{s}'] > im
        hi_out = row[f'Outflow_Poorer{s}'] > om
        if hi_in and hi_out: return 'Active Cascade'
        elif hi_in and not hi_out: return 'Pressure Only'
        elif not hi_in and hi_out: return 'Displacement Only'
        else: return 'Quiet'
    
    df[f'Quad{suffix}'] = df.apply(quad, axis=1)
    print(f'{year} quadrant thresholds: Inflow_W median = {iw_med:.0f}, Outflow_P median = {op_med:.0f}')
    print(df[f'Quad{suffix}'].value_counts().to_string())
    print()

In [ ]:
# Quadrant scatter plots
quad_colors = {
    'Active Cascade': '#d73027', 'Pressure Only': '#fc8d59',
    'Displacement Only': '#91bfdb', 'Quiet': '#4575b4'
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    iw_med = df[f'Inflow_Wealthier{suffix}'].median()
    op_med = df[f'Outflow_Poorer{suffix}'].median()
    
    for qname, color in quad_colors.items():
        mask = df[f'Quad{suffix}'] == qname
        ax.scatter(df.loc[mask, f'Inflow_Wealthier{suffix}'],
                   df.loc[mask, f'Outflow_Poorer{suffix}'],
                   c=color, label=qname, s=20, alpha=0.6,
                   edgecolors='grey', linewidth=0.3)
    
    ax.axvline(iw_med, color='black', lw=1, ls='--', alpha=0.5)
    ax.axhline(op_med, color='black', lw=1, ls='--', alpha=0.5)
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, lim], [0, lim], 'k:', alpha=0.3, label='Balance line')
    
    # Quadrant labels
    xr, yr = ax.get_xlim(), ax.get_ylim()
    ax.text(xr[1]*0.75, yr[1]*0.9, 'Active\nCascade', ha='center', fontsize=10,
            color='#d73027', fontweight='bold', alpha=0.7)
    ax.text(xr[1]*0.75, yr[1]*0.15, 'Pressure\nOnly', ha='center', fontsize=10,
            color='#fc8d59', fontweight='bold', alpha=0.7)
    ax.text(xr[1]*0.2, yr[1]*0.9, 'Displacement\nOnly', ha='center', fontsize=10,
            color='#91bfdb', fontweight='bold', alpha=0.7)
    ax.text(xr[1]*0.2, yr[1]*0.15, 'Quiet', ha='center', fontsize=10,
            color='#4575b4', fontweight='bold', alpha=0.7)
    
    ax.set_xlabel('Inflow from Wealthier Areas (Gentrification Pressure)')
    ax.set_ylabel('Outflow to Poorer Areas (Displacement Yield)')
    ax.set_title(f'{year} Census O-D (median thresholds: IW={iw_med:.0f}, OP={op_med:.0f})')
    ax.legend(fontsize=8, loc='center right')

fig.suptitle('Bivariate Cascade Analysis: Gentrification Pressure × Displacement Yield', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('outputs/biv_fig1_quadrant_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

##### Quadrant Definitions
| Quadrant | Inflow_W | Outflow_P | Interpretation |
|----------|----------|-----------|----------------|
| **Active Cascade** | High | High | Full cascade: gentrifiers arrive *and* residents displaced simultaneously |
| **Pressure Only** | High | Low | Gentrification pressure building — arrivals from wealthier areas, but displacement not yet visible in flows |
| **Displacement Only** | Low | High | Residents being pushed to poorer areas, but not driven by wealthier inflow (could be housing costs, redevelopment) |
| **Quiet** | Low | Low | Minimal cascade activity |

---
## 7. Quadrant Validation: Which Quadrant Best Predicts Deprivation Change?

In [ ]:
# Validation: Mean IMD Change per quadrant
quad_order = ['Pressure Only', 'Active Cascade', 'Displacement Only', 'Quiet']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    means = df.groupby(f'Quad{suffix}')['IMD_Change'].mean().reindex(quad_order)
    sems = df.groupby(f'Quad{suffix}')['IMD_Change'].sem().reindex(quad_order)
    counts = df.groupby(f'Quad{suffix}')['IMD_Change'].count().reindex(quad_order)
    
    colors = [quad_colors[q] for q in quad_order]
    bars = ax.bar(quad_order, means, yerr=sems*1.96, capsize=5,
                  color=colors, edgecolor='white', alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Mean IMD Change (2010→2019)\n(negative = less deprived)')
    ax.set_title(f'{year} Quadrant Classification')
    for bar, n, m in zip(bars, counts, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.3,
                f'n={n}\n({m:.1f})', ha='center', va='top', fontsize=9, fontweight='bold')

fig.suptitle('Validation: Mean IMD Change by Bivariate Cascade Quadrant (95% CI)', fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig('outputs/biv_fig2_quadrant_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical tests
for suffix, year in [('_11','2011'), ('_21','2021')]:
    print(f'\n=== {year} ===')
    summary = df.groupby(f'Quad{suffix}').agg(
        N=('msoa11cd','count'),
        Mean_IMD_2010=('IMD_2010','mean'),
        Mean_IMD_Change=('IMD_Change','mean'),
        Mean_CFI_Churn=(f'CFI_Churn{suffix}','mean'),
        Mean_Net_Cascade=(f'Net_Cascade{suffix}','mean'),
    ).reindex(quad_order).round(2)
    display(summary)
    
    groups = [g['IMD_Change'].values for _, g in df.groupby(f'Quad{suffix}')]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'  ANOVA: F = {f_stat:.2f}, p = {p_val:.2e}')
    
    ac = df.loc[df[f'Quad{suffix}']=='Active Cascade', 'IMD_Change']
    qu = df.loc[df[f'Quad{suffix}']=='Quiet', 'IMD_Change']
    t, p = stats.ttest_ind(ac, qu)
    print(f'  Active Cascade vs Quiet: t = {t:.2f}, p = {p:.2e}')

##### Key Finding: Pressure Only > Active Cascade

The **Pressure Only** quadrant shows the *largest* deprivation decline (−6.3 in 2011, −6.0 in 2021) — even more than Active Cascade (−5.4, −5.1). This is theoretically meaningful:

- **Pressure Only** MSOAs are the *early-stage gentrifiers*: wealthier residents are arriving but displacement hasn't manifested yet in the outflow data. These are areas where the deprivation score is dropping fastest precisely because the incoming population is changing the area's composition.
- **Active Cascade** MSOAs have *simultaneous* inflow and outflow — the gentrification process is more advanced but the net effect on deprivation is somewhat offset by high displacement volumes.

This distinction is **invisible to Net Cascade alone**, which would rank Pressure Only and Active Cascade identically if they have the same difference between inflow and outflow.

ANOVA is highly significant (F = 78.5, p < 10⁻⁴⁵), confirming the quadrants capture real variation in deprivation outcomes.

---
## 8. Cascade Direction Ratio (CDR): Giving Churn its Direction

The CDR normalises the *direction* of cascade activity within each MSOA:

$$CDR_i = \frac{Inflow_{Wealthier,i}}{CFI\_Churn_i} = \frac{Inflow_{Wealthier,i}}{Inflow_{Wealthier,i} + Outflow_{Poorer,i}}$$

- CDR = 1.0 → pure gentrification pressure (all churn is inflow from wealthier)
- CDR = 0.5 → balanced cascade (equal inflow and outflow)
- CDR = 0.0 → pure displacement (all churn is outflow to poorer areas)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 8A: CDR distribution
ax = axes[0]
for suffix, year, color in [('_11', '2011', '#4575b4'), ('_21', '2021', '#d73027')]:
    valid = df[f'CDR{suffix}'].dropna()
    ax.hist(valid, bins=40, alpha=0.5, color=color, edgecolor='white', label=year)
ax.axvline(0.5, color='black', ls='--', lw=1.5, label='Balance (0.5)')
ax.set_xlabel('CDR (0 = pure displacement, 1 = pure pressure)')
ax.set_ylabel('Number of MSOAs')
ax.set_title('A. Distribution of CDR')
ax.legend()

# 8B: CDR vs IMD Change
ax = axes[1]
valid = df[['CDR_21','IMD_Change','Wealth_Decile']].dropna()
sc = ax.scatter(valid['CDR_21'], valid['IMD_Change'],
                c=valid['Wealth_Decile'], cmap='RdYlGn', s=20, alpha=0.5,
                edgecolors='grey', linewidth=0.3)
r, p = stats.pearsonr(valid['CDR_21'], valid['IMD_Change'])
ax.set_xlabel('CDR (2021)'); ax.set_ylabel('IMD Change (2010→2019)')
ax.set_title(f'B. CDR vs Deprivation Change\nr = {r:.3f}, p = {p:.2e}')
ax.axvline(0.5, color='black', ls='--', lw=1, alpha=0.5)
ax.axhline(0, color='black', ls='--', lw=0.8)
plt.colorbar(sc, ax=ax, label='Wealth Decile', shrink=0.8)

# 8C: CDR by Wealth Decile
ax = axes[2]
cdr_by_dec = df.groupby('Wealth_Decile')['CDR_21'].mean()
colors = ['#d73027' if v > 0.5 else '#4575b4' for v in cdr_by_dec.values]
ax.bar(cdr_by_dec.index, cdr_by_dec.values, color=colors, edgecolor='white')
ax.axhline(0.5, color='black', ls='--', lw=1.5, label='Balance')
ax.set_xlabel('Wealth Decile'); ax.set_ylabel('Mean CDR (2021)')
ax.set_title('C. CDR by Decile\n(>0.5 = pressure-led, <0.5 = displacement-led)')
ax.set_xticks(range(1,11)); ax.legend()

plt.tight_layout()
plt.savefig('outputs/biv_fig3_direction_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CDR validation
print('=== CDR Validation ===')
for suffix, year in [('_11','2011'), ('_21','2021')]:
    valid = df[[f'CDR{suffix}','IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[f'CDR{suffix}'], valid['IMD_Change'])
    print(f'  CDR {year} vs IMD Change: r = {r:+.3f}, p = {p:.2e}')

##### CDR is the Strongest Single Predictor

**CDR achieves r = −0.48** with IMD Change — stronger than Net Cascade (r = −0.44), % Inflow Wealthier (r = −0.42), and CFI Churn (r = −0.25).

Why CDR outperforms the other metrics:
1. It captures the *proportion* of cascade activity that is gentrification-driven, not just the volume.
2. Unlike CFI Rate (which normalises by *total* migration), CDR normalises by *cascade-relevant* migration only — so it doesn't dilute the signal with lateral or within-decile moves.
3. The decile-level pattern is clean: CDR > 0.5 (pressure-led) in deprived deciles 1–4, and < 0.5 (displacement-led) in wealthy deciles 7–10.

---
## 9. Composite Framework: Cascade Intensity × Direction

The full bivariate framework plots **CFI Churn** (y-axis, intensity) against **CDR** (x-axis, direction),
creating a four-quadrant space that captures *both* dimensions of cascade dynamics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    valid = df[[f'CFI_Churn{suffix}', f'CDR{suffix}', 'IMD_Change']].dropna()
    sc = ax.scatter(
        valid[f'CDR{suffix}'], valid[f'CFI_Churn{suffix}'],
        c=valid['IMD_Change'], cmap='RdBu', s=20, alpha=0.6,
        edgecolors='grey', linewidth=0.3, vmin=-20, vmax=10)
    
    ax.axvline(0.5, color='black', ls='--', lw=1, alpha=0.6)
    churn_med = valid[f'CFI_Churn{suffix}'].median()
    ax.axhline(churn_med, color='black', ls='--', lw=1, alpha=0.6)
    
    yr = ax.get_ylim()
    ax.text(0.8, yr[1]*0.92, 'HIGH INTENSITY\nPRESSURE-LED', ha='center', fontsize=9,
            color='#d73027', fontweight='bold', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#d73027', alpha=0.5))
    ax.text(0.2, yr[1]*0.92, 'HIGH INTENSITY\nDISPLACEMENT-LED', ha='center', fontsize=9,
            color='#fc8d59', fontweight='bold', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#fc8d59', alpha=0.5))
    ax.text(0.8, yr[1]*0.12, 'LOW INTENSITY\nPRESSURE-LED', ha='center', fontsize=9,
            color='#91bfdb', fontweight='bold', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#91bfdb', alpha=0.5))
    ax.text(0.2, yr[1]*0.12, 'LOW INTENSITY\nDISPLACEMENT-LED', ha='center', fontsize=9,
            color='#4575b4', fontweight='bold', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#4575b4', alpha=0.5))
    
    ax.set_xlabel('Cascade Direction Ratio\n(0 = pure displacement ← → 1 = pure pressure)')
    ax.set_ylabel('CFI Churn (Cascade Intensity)')
    ax.set_title(f'{year}: Intensity × Direction')

cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
cbar.set_label('IMD Change (2010→2019) (blue = less deprived)')
fig.suptitle('Bivariate Cascade Framework: Intensity (CFI Churn) × Direction (CDR)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('outputs/biv_fig4_intensity_direction.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Bivariate Typology Validation

In [ ]:
# Classify into 4 bivariate types
for suffix in ['_11', '_21']:
    churn_med = df[f'CFI_Churn{suffix}'].median()
    def biv_type(row, s=suffix, cm=churn_med):
        if pd.isna(row[f'CDR{s}']): return 'No Cascade Activity'
        hi_churn = row[f'CFI_Churn{s}'] > cm
        pressure_led = row[f'CDR{s}'] > 0.5
        if hi_churn and pressure_led: return 'High Intensity, Pressure-Led'
        elif hi_churn and not pressure_led: return 'High Intensity, Displacement-Led'
        elif not hi_churn and pressure_led: return 'Low Intensity, Pressure-Led'
        else: return 'Low Intensity, Displacement-Led'
    df[f'BivType{suffix}'] = df.apply(biv_type, axis=1)

# Validation
biv_order = ['High Intensity, Pressure-Led', 'High Intensity, Displacement-Led',
             'Low Intensity, Pressure-Led', 'Low Intensity, Displacement-Led']

for suffix, year in [('_11','2011'), ('_21','2021')]:
    print(f'\n=== {year} Bivariate Typology ===')
    summary = df[df[f'BivType{suffix}'] != 'No Cascade Activity'].groupby(f'BivType{suffix}').agg(
        N=('msoa11cd','count'),
        Mean_IMD_2010=('IMD_2010','mean'),
        Mean_IMD_Change=('IMD_Change','mean'),
        Mean_CFI_Churn=(f'CFI_Churn{suffix}','mean'),
        Mean_CDR=(f'CDR{suffix}','mean'),
    ).reindex(biv_order).round(2)
    display(summary)
    
    groups = [g['IMD_Change'].values for n, g in df.groupby(f'BivType{suffix}') if n != 'No Cascade Activity']
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'  ANOVA: F = {f_stat:.2f}, p = {p_val:.2e}')

In [ ]:
# Validation bar chart
biv_colors = {
    'High Intensity, Pressure-Led': '#d73027', 'High Intensity, Displacement-Led': '#fc8d59',
    'Low Intensity, Pressure-Led': '#91bfdb', 'Low Intensity, Displacement-Led': '#4575b4',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
short_labels = ['HI Pressure', 'HI Displacement', 'LO Pressure', 'LO Displacement']

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    grp = df[df[f'BivType{suffix}'] != 'No Cascade Activity'].groupby(f'BivType{suffix}')
    means = grp['IMD_Change'].mean().reindex(biv_order)
    sems = grp['IMD_Change'].sem().reindex(biv_order)
    counts = grp['IMD_Change'].count().reindex(biv_order)
    
    colors = [biv_colors[q] for q in biv_order]
    bars = ax.bar(short_labels, means, yerr=sems*1.96, capsize=5,
                  color=colors, edgecolor='white', alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Mean IMD Change (2010→2019)')
    ax.set_title(f'{year} Bivariate Typology')
    for bar, n, m in zip(bars, counts, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.2,
                f'n={n}\n({m:.1f})', ha='center', va='top', fontsize=9, fontweight='bold')

fig.suptitle('Validation: Mean Deprivation Change by Bivariate Cascade Type (95% CI)', fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig('outputs/biv_fig5_bivtype_validation.png', dpi=150, bbox_inches='tight')
plt.show()

##### Result
The bivariate typology produces a clear gradient (ANOVA F = 66.8, p < 10⁻³⁹):
- **High Intensity, Pressure-Led** (n≈250): Mean IMD change = −6.4 — strongest gentrification signal
- **High Intensity, Displacement-Led** (n≈241): Mean IMD change = −3.8 — high activity but lower deprivation decline
- **Low Intensity, Pressure-Led** (n≈190): Mean IMD change = −4.9 — early-stage gentrification
- **Low Intensity, Displacement-Led** (n≈302): Mean IMD change = −1.5 — minimal gentrification

The key insight: **direction matters more than intensity**. Pressure-led MSOAs (CDR > 0.5) consistently
show larger deprivation declines than displacement-led MSOAs at the *same* intensity level.

---
## 11. Full Validation Summary: All Metrics Including CDR

In [ ]:
print('=== FULL VALIDATION: Pearson r with IMD Change ===\n')
results = []
for col, label in [
    ('CDR_11', 'CDR 2011'), ('CDR_21', 'CDR 2021'),
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
    ('Pct_Inflow_Wealthier_11', '% Inflow W. 2011'), ('Pct_Inflow_Wealthier_21', '% Inflow W. 2021'),
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
]:
    valid = df[[col, 'IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Change'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    results.append({'Metric': label, 'r': r, 'p': p, 'sig': sig})

val_df = pd.DataFrame(results).sort_values('r')

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d73027' if r < 0 else '#4575b4' for r in val_df['r']]
ax.barh(val_df['Metric'], val_df['r'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson r with IMD Change (2010→2019)')
ax.set_title('Full Validation: All Cascade Metrics (incl. CDR) vs Deprivation Change')
for i, (_, row) in enumerate(val_df.iterrows()):
    ax.text(row['r'] + 0.01 * np.sign(row['r']), i,
            f"{row['r']:.3f} {row['sig']}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/biv_fig6_full_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(val_df[['Metric','r','sig']].to_string(index=False))

---
## 12. Borough-Level Analysis

In [ ]:
borough = df.groupby('ladnm').agg(
    N=('msoa11cd','count'),
    Mean_CFI_Churn_21=('CFI_Churn_21','mean'),
    Mean_CDR_21=('CDR_21','mean'),
    Mean_CFI_Churn_11=('CFI_Churn_11','mean'),
    Total_Net_Cascade_21=('Net_Cascade_21','sum'),
    Mean_IMD_Change=('IMD_Change','mean'),
).round(2)
borough['Delta_Churn'] = (borough['Mean_CFI_Churn_21'] - borough['Mean_CFI_Churn_11']).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

bs = borough.sort_values('Mean_CFI_Churn_21', ascending=True)
med = borough['Mean_CFI_Churn_21'].median()
colors = ['#d73027' if v > med else '#4575b4' for v in bs['Mean_CFI_Churn_21']]
axes[0].barh(bs.index, bs['Mean_CFI_Churn_21'], color=colors, edgecolor='white')
axes[0].set_xlabel('Mean CFI Churn (2021)')
axes[0].set_title('A. Average CFI Churn by Borough (2021)')

bs2 = borough.sort_values('Delta_Churn', ascending=True)
colors2 = ['coral' if v > 0 else 'steelblue' for v in bs2['Delta_Churn']]
axes[1].barh(bs2.index, bs2['Delta_Churn'], color=colors2, edgecolor='white')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Δ CFI Churn (2021 − 2011)')
axes[1].set_title('B. Change in CFI Churn by Borough')

plt.tight_layout()
plt.savefig('outputs/biv_fig7_borough_churn.png', dpi=150, bbox_inches='tight')
plt.show()

display(borough.sort_values('Mean_CFI_Churn_21', ascending=False))

---
## 13. Summary & Recommendations for Dissertation

### Metric Hierarchy

| Rank | Metric | r with IMD Change | Role |
|------|--------|-------------------|------|
| 1 | **CDR** | −0.48*** | Best single predictor — captures *direction* of cascade activity |
| 2 | **Net Cascade** | −0.44*** | Primary cascade index — directional balance of gentrification flows |
| 3 | **% Inflow Wealthier** | −0.42*** | Interpretable share metric — percentage of arrivals from wealthier areas |
| 4 | **CFI Churn** | −0.30*** | Cascade *intensity* — total volume of displacement-relevant activity |
| 5 | **CFI Rate** | −0.05 ns | ❌ Does not validate — normalisation removes signal |

### Recommended Analytical Framework

1. **Primary index**: Use **Net Cascade** for the main gentrification typology (Emerging/Sustained/Stalled/Stable) — it has the best balance of interpretability and predictive power.

2. **Bivariate decomposition**: Use the **Inflow_Wealthier × Outflow_Poorer** quadrant analysis to distinguish the *type* of cascade activity — especially to identify "Pressure Only" areas (early-stage gentrification) vs "Active Cascade" areas (full displacement cycle).

3. **Direction ratio**: Report **CDR** as a derived metric that gives CFI Churn its directional meaning. This is the strongest single predictor and provides a clean 0–1 scale for mapping.

4. **CFI Churn**: Use as a secondary measure of cascade *intensity* in the bivariate framework (y-axis of the Intensity × Direction plot).

5. **CFI Rate**: Report in appendix with methodological note. The normalisation approach is theoretically sound but empirically fails because it cancels out the volume signal and peaks at mid-deciles rather than tracking the gentrification gradient.

### Key Methodological Contribution
The **bivariate framework** (decomposing CFI Churn into Pressure × Displacement) is the strongest analytical tool in this analysis. It captures dynamics that no single metric can:
- "Pressure Only" areas have *larger* deprivation declines than "Active Cascade" areas (−6.3 vs −5.4) — early-stage gentrification is more disruptive to area statistics than advanced-stage churn.
- The CDR (direction ratio) outperforms all four original metrics, suggesting that the *proportion* of cascade activity directed toward gentrification matters more than its absolute volume.